In [ ]:
import pandas as pd


grouped = pd.read_csv('data/matched_max.csv', sep="\t")
# TODO: FIX
unique_counts = grouped['discogs_attr'].nunique()
filtered = unique_counts[unique_counts >= 2]

# To avoid one video mapping to multiple cliques
filtered = filtered[filtered.index.duplicated(keep=False) == False]
print("Total:", len(filtered))


TypeError: 'int' object is not subscriptable

In [2]:
grouped

,youtube_id,clique_id,youtube_attr,discogs_attr,Score
0,Z88g_AhoFKE,C-0210063,video_title,track_title_cleaned,100.000000
1,Z88g_AhoFKE,C-0210063,description,track_title_cleaned,100.000000
2,Z88g_AhoFKE,C-0210063,video_title,release_artist_names,28.571428
3,Z88g_AhoFKE,C-0210063,description,release_artist_names,28.571428
4,Z88g_AhoFKE,C-0090996,video_title,track_title_cleaned,92.307690
...,...,...,...,...,...
29504835,OnxicyxiTxA,C-0285302,description,release_artist_names,0.000000
29504836,BT4iSgtIHLI,C-0285302,video_title,track_title_cleaned,100.000000
29504837,BT4iSgtIHLI,C-0285302,description,track_title_cleaned,69.565216
29504838,BT4iSgtIHLI,C-0285302,video_title,release_artist_names,33.333332


In [ ]:
metadata = pd.read_json("data/metadata_filtered.jsonl", lines=True, orient='records')
metadata = metadata.loc[metadata.id.isin(filtered.index.get_level_values(0)),:]

new_columns = ["clique_id"] + metadata.columns.tolist()
metadata = pd.merge(
    filtered.to_frame().reset_index().drop("discogs_attr", axis=1),
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)
metadata = metadata[new_columns]


In [ ]:
def time_to_seconds(time_str):
    parts = list(map(int, time_str.split(":")))
    if len(parts) == 3:  # HH:MM:SS
        h, m, s = parts
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:  # MM:SS
        m, s = parts
        return m * 60 + s
    elif len(parts) == 1:  # SS
        return parts[0]
    else:
        raise ValueError(f"Unrecognized time format: {time_str}")
    
metadata["duration_secs"] = metadata["duration"].apply(time_to_seconds)
metadata["duration_mins"] = metadata["duration_secs"] / 60

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.displot(metadata, x="duration_mins", bins=100)
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of videos (log scaled)")
plt.yscale('log')  # <-- set y-axis to log scale
plt.show()

In [ ]:
def get_description_str(descriptionSnippet):
    if isinstance(descriptionSnippet, list):
        return " ".join([d["text"] for d in descriptionSnippet]).replace("\n", " ").replace("\r", " ")
    else:
        return descriptionSnippet

metadata["description"] = metadata.descriptionSnippet.apply(get_description_str)


In [ ]:
import pandas as pd
from collections import Counter
from itertools import islice
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# Make sure you've downloaded stopwords & tokenizer models:
# nltk.download('punkt')
# nltk.download('stopwords')

# Combine stopwords from multiple languages
languages = ['english', 'german', 'french', 'spanish', 'italian', 'portuguese']  # extend as needed
stop_words = set()
for lang in languages:
    stop_words.update(stopwords.words(lang))

def ngrams(tokens, n):
    return zip(*(islice(tokens, i, None) for i in range(n)))

def clean_and_tokenize(text):
    tokens = word_tokenize(text.lower())
    return [word for word in tokens if word.isalnum() and word not in stop_words]

def get_top_ngrams(series, n=1, top_k=20):
    all_ngrams = Counter()
    for text in series.dropna().astype(str):
        tokens = clean_and_tokenize(text)
        all_ngrams.update(ngrams(tokens, n))
    return all_ngrams.most_common(top_k)

# Assuming your dataframe is called df
for column in ['title', 'description']:
    print(f"\n==== {column.upper()} ====")
    for n in [1, 2, 3]:
        top = get_top_ngrams(metadata[column], n=n, top_k=15)  # adjust top_k as needed
        print(f"\nTop {n}-grams (stopwords removed):")
        for phrase, count in top:
            joined = " ".join(phrase) if isinstance(phrase, tuple) else phrase
            print(f"{joined}: {count}")
